In [1]:
pip install pyfaidx

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [3]:
"""
NeuroSight M5 — Synthetic cfDNA Dataset Preparation Pipeline
=============================================================
Converts real patient DNA methylation arrays (TCGA / GEO) into synthetic
plasma cfDNA fragments in Pleiades-compatible JSON format.

Rules implemented per: Synthetic_GBM_cfDNA_Rulebook.pdf (Arnav Mishra, 2025)
Citations: Nassiri et al. 2020, Niki et al. (Pleiades) 2025, Mouliere et al. 2018,
           Moss et al. 2018, Sanchez et al. 2021

Author: Arnav Mishra
Project: NeuroSight — NeuroBio Model 5 (M5)

Dataset decisions (4-class setup):
  Label 0 — Healthy   : GSE40279 (whole-blood methylation)
  Label 1 — GBM       : TCGA-GBM SeSAMe Level 3 betas
  Label 2 — LGG       : TCGA-LGG SeSAMe Level 3 betas (single class, no clinical split)
  Label 3 — DMG_H3K27M: GSE161944 (adult H3K27M, ~20 real samples) → augmented to 200

FIXES APPLIED (vs original):
  FIX-2  Rule 7  — ctDNA fraction metadata corrected: JSON now stores both
                   raw_ctdna_fraction (biological ground truth, ~3.1e-5) and
                   enriched_ctdna_fraction (actual mixing ratio post-cfMeDIP,
                   5-30%).  Previously only raw was stored while enriched was used.
  FIX-3  Rule 7  — Healthy pool beta lookup: per-probe beta values are now looked
                   up by manifest cg_id at the exact genomic position, matching
                   the same logic used for tumor patients.  Previously the entire
                   column mean was assigned to every CpG, collapsing all positional
                   methylation variation into a single scalar.
"""

import os
import json
import glob
import logging
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import beta as beta_dist
from pyfaidx import Fasta
from tqdm import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

BETA_HIGH = 0.70
BETA_LOW  = 0.30

N_DMRS      = 300
DELTA_BETA  = 0.30
MIN_CPGS    = 2

FRAG_HEALTHY_MONO   = 166
FRAG_TUMOR_MONO     = 139
FRAG_DINU           = 320
FRAG_SD_HEALTHY     = 15
FRAG_SD_TUMOR       = 12
FRAG_MONO_RATIO     = 0.75
FRAG_MIN            = 100
FRAG_MAX            = 400
FRAG_TUMOR_MONO_MIN = 134
FRAG_TUMOR_MONO_MAX = 160

MIN_CPG_PER_FRAG = 2
MAX_CPG_PER_FRAG = 8

CTDNA_MEDIAN      = 3.1e-5
CTDNA_RANGE_MAX   = 1e-4
ENRICHED_CTDNA_MIN = 0.05
ENRICHED_CTDNA_MAX = 0.30

FRAGS_PER_PATIENT_MIN = 500
FRAGS_PER_PATIENT_MAX = 1000

WINDOW_BP = 300

DMG_AUGMENT_TARGET = 200

# 4-class label map — no 5th class
LABEL_MAP = {
    0: "Healthy",
    1: "GBM",
    2: "LGG",
    3: "DMG_H3K27M",
}

HEALTHY_CELL_FRACTIONS = {
    "WBC":         0.55,
    "RBC_prog":    0.30,
    "Endothelium": 0.10,
    "Hepatocyte":  0.01,
    "Other":       0.04,
}

# ---------------------------------------------------------------------------
# Data Structures
# ---------------------------------------------------------------------------

@dataclass
class DMR:
    chromosome: str
    start: int
    end: int
    cpg_ids: List[str]
    delta_beta: float
    t_stat: float
    label: int


@dataclass
class Fragment:
    tokens: List[str]
    fragment_length: int
    cpg_count: int
    strand: str
    genomic_start: int
    chromosome: str


# ---------------------------------------------------------------------------
# Step 1 — Data Loading
# ---------------------------------------------------------------------------

class DataLoader:

    def load_tcga_per_patient(
        self,
        root_dir: str,
        label: int,
        na_threshold: float = 0.70,
    ) -> pd.DataFrame:
        """
        Load TCGA per-patient files (one .txt per UUID subfolder).
        Each file: no header, tab-sep, col0=cg_id, col1=beta_value.
        NAs ~67% per file are expected for SeSAMe Level 3 processed data.
        """
        log.info(f"Loading TCGA data from {root_dir} (label={label})")
        txt_files = glob.glob(
            os.path.join(root_dir, "**", "*.methylation_array.sesame.level3betas.txt"),
            recursive=True,
        )
        if not txt_files:
            raise FileNotFoundError(
                f"No TCGA methylation files found under {root_dir}. "
                "Expected: <UUID>/<UUID>.methylation_array.sesame.level3betas.txt"
            )

        frames = []
        for fpath in tqdm(txt_files, desc=f"TCGA label={label}"):
            patient_id = Path(fpath).stem
            df = pd.read_csv(
                fpath, sep="\t", header=None,
                names=["cg_id", patient_id], index_col="cg_id",
            )
            frames.append(df)

        matrix = pd.concat(frames, axis=1)
        log.info(f"  Raw shape: {matrix.shape}")

        na_rate = matrix.isna().mean(axis=1)
        matrix  = matrix.loc[na_rate <= na_threshold]
        log.info(
            f"  After NA filter ({na_threshold:.0%}): {matrix.shape} "
            f"({(na_rate > na_threshold).sum()} probes dropped)"
        )
        matrix = matrix.apply(lambda col: col.fillna(col.median()), axis=1)
        log.info(f"  Beta mean={matrix.values.mean():.3f}  std={matrix.values.std():.3f}")
        return matrix

    def load_geo_matrix(
        self,
        filepath: str,
        id_col: str = "ID_REF",
    ) -> pd.DataFrame:
        """
        Load a GEO matrix file (probes x samples).
        Handles GSE161944 (850K DMG) and GSE40279 (450K healthy).
        """
        log.info(f"Loading GEO matrix: {filepath}")
        df = pd.read_csv(filepath, sep="\t", index_col=id_col, low_memory=False)
        df = df.apply(pd.to_numeric, errors="coerce")
        na_rate = df.isna().mean(axis=1)
        df = df.loc[na_rate < 0.80]
        df = df.apply(lambda col: col.fillna(col.median()), axis=1)
        log.info(f"  Shape: {df.shape}  Beta mean={df.values.mean():.3f}")
        return df


# ---------------------------------------------------------------------------
# Step 2 — Common CpG Intersection
# ---------------------------------------------------------------------------

def compute_common_cpg_space(*matrices: pd.DataFrame) -> List[str]:
    common = set(matrices[0].index)
    for m in matrices[1:]:
        common &= set(m.index)
    common = sorted(common)
    log.info(f"Common CpG space: {len(common):,} probes")
    return common


# ---------------------------------------------------------------------------
# Step 3 — DMG Augmentation via Beta Distribution Fitting
# ---------------------------------------------------------------------------

class DMGAugmentor:
    """
    Augments the small DMG H3K27M cohort (GSE161944, ~20 real samples) to
    DMG_AUGMENT_TARGET by fitting independent Beta distributions per probe.
    """

    def fit(self, matrix: pd.DataFrame) -> None:
        log.info(
            f"Fitting Beta distributions for DMG augmentation "
            f"({matrix.shape[1]} real samples, GSE161944 only) ..."
        )
        self._params: Dict[str, Tuple[float, float]] = {}
        for probe_id, row in tqdm(matrix.iterrows(), total=len(matrix), desc="Beta fit"):
            vals = row.dropna().values.clip(1e-6, 1 - 1e-6)
            if len(vals) < 3:
                mu = float(np.mean(vals)) if len(vals) else 0.5
                self._params[probe_id] = (mu * 10, (1 - mu) * 10)
                continue
            try:
                a, b, _, _ = beta_dist.fit(vals, floc=0, fscale=1)
                self._params[probe_id] = (a, b)
            except Exception:
                mu = float(np.mean(vals))
                self._params[probe_id] = (mu * 10, (1 - mu) * 10)

    def sample(self, n: int = DMG_AUGMENT_TARGET) -> pd.DataFrame:
        log.info(f"Sampling {n} synthetic DMG patients ...")
        data = {
            probe_id: beta_dist.rvs(a, b, size=n).clip(0, 1)
            for probe_id, (a, b) in self._params.items()
        }
        df = pd.DataFrame(data).T
        df.columns = [f"DMG_synthetic_{i:04d}" for i in range(n)]
        log.info(f"  Augmented DMG matrix: {df.shape}")
        return df


# ---------------------------------------------------------------------------
# Step 4 — DMR Identification
# ---------------------------------------------------------------------------

class DMRFinder:
    """
    Memory-efficient DMR finder.
    Processes one class at a time — never concatenates all matrices at once.
    Subsamples to MAX_SAMPLES per class before t-test to save RAM.
    """

    MAX_SAMPLES_PER_CLASS = 80

    def __init__(self, cpg_manifest: pd.DataFrame):
        self.manifest = cpg_manifest

    def find_dmrs(
        self,
        matrices: Dict[int, pd.DataFrame],
        n_dmrs: int = N_DMRS,
        delta_beta_thresh: float = DELTA_BETA,
        min_cpgs: int = MIN_CPGS,
    ) -> Dict[int, List[DMR]]:
        log.info("Finding DMRs (memory-efficient one-vs-rest t-test) ...")

        # ── Subsample each class to MAX_SAMPLES to save RAM ──
        rng = np.random.default_rng(42)
        subsampled: Dict[int, np.ndarray] = {}
        cpg_index = None

        for lbl, mat in matrices.items():
            n = mat.shape[1]
            if n > self.MAX_SAMPLES_PER_CLASS:
                idx = rng.choice(n, self.MAX_SAMPLES_PER_CLASS, replace=False)
                sub = mat.iloc[:, idx].values.astype(np.float32)
            else:
                sub = mat.values.astype(np.float32)
            subsampled[lbl] = sub
            if cpg_index is None:
                cpg_index = mat.index.tolist()
            log.info(f"  Label {lbl} ({LABEL_MAP[lbl]}): using {sub.shape[1]} samples")

        del matrices  # free original matrices from RAM
        import gc; gc.collect()

        cpg_array = np.array(cpg_index)
        dmrs_by_label: Dict[int, List[DMR]] = {}

        for target_label in sorted(subsampled.keys()):
            log.info(f"  Computing DMRs for label {target_label} ({LABEL_MAP[target_label]}) ...")

            pos_mat = subsampled[target_label]  # (n_cpgs, n_pos)

            # Stack negatives one at a time to avoid one giant concat
            neg_parts = [subsampled[lbl] for lbl in subsampled if lbl != target_label]
            neg_mat   = np.concatenate(neg_parts, axis=1)  # (n_cpgs, n_neg)

            # t-test row-wise (per CpG)
            t_stats, p_vals = stats.ttest_ind(
                pos_mat, neg_mat, axis=1, equal_var=False, nan_policy="omit"
            )
            delta = np.nanmean(pos_mat, axis=1) - np.nanmean(neg_mat, axis=1)

            del neg_mat; gc.collect()

            # BH correction + filter
            valid    = ~(np.isnan(t_stats) | np.isnan(p_vals))
            t_stats  = t_stats[valid]
            p_vals   = p_vals[valid]
            delta    = delta[valid]
            cpgs_v   = cpg_array[valid]

            sort_idx = np.argsort(p_vals)
            t_stats  = t_stats[sort_idx]
            p_vals   = p_vals[sort_idx]
            delta    = delta[sort_idx]
            cpgs_v   = cpgs_v[sort_idx]

            n        = len(p_vals)
            ranks    = np.arange(1, n + 1)
            bh_thresh = ranks / n * 0.01
            keep     = (p_vals <= bh_thresh) & (np.abs(delta) >= delta_beta_thresh)

            t_stats_f = t_stats[keep]
            p_vals_f  = p_vals[keep]
            delta_f   = delta[keep]
            cpgs_f    = cpgs_v[keep]

            log.info(f"    Significant CpGs after BH: {keep.sum():,}")

            # Join with manifest for coordinates
            result_df = pd.DataFrame({
                "cg_id":      cpgs_f,
                "t_stat":     t_stats_f,
                "p_val":      p_vals_f,
                "delta_beta": delta_f,
            }).set_index("cg_id")

            result_df = result_df.join(self.manifest, how="inner")

            dmr_list  = self._group_into_windows(result_df, target_label, min_cpgs)
            dmr_list  = sorted(dmr_list, key=lambda d: abs(d.t_stat), reverse=True)[:n_dmrs]
            dmrs_by_label[target_label] = dmr_list
            log.info(f"    Label {target_label}: {len(dmr_list)} DMRs found")

            del result_df; gc.collect()

        return dmrs_by_label

    def _group_into_windows(
        self,
        result_df: pd.DataFrame,
        label: int,
        min_cpgs: int,
    ) -> List[DMR]:
        dmrs = []
        for chrom, grp in result_df.groupby("chromosome"):
            grp  = grp.sort_values("start")
            used = np.zeros(len(grp), dtype=bool)
            for i, (idx, row) in enumerate(grp.iterrows()):
                if used[i]:
                    continue
                window_start = int(row["start"])
                window_end   = window_start + WINDOW_BP
                in_window    = (
                    (grp["start"] >= window_start) & (grp["start"] < window_end)
                ).values
                if in_window.sum() < min_cpgs:
                    continue
                window_cpgs = grp[in_window]
                dmrs.append(DMR(
                    chromosome=str(chrom),
                    start=window_start,
                    end=window_end,
                    cpg_ids=window_cpgs.index.tolist(),
                    delta_beta=float(window_cpgs["delta_beta"].mean()),
                    t_stat=float(window_cpgs["t_stat"].mean()),
                    label=label,
                ))
                used[in_window] = True
        return dmrs  

# ---------------------------------------------------------------------------
# Step 5 — Sequence Extraction + Beta -> Token Conversion
# ---------------------------------------------------------------------------

class SequenceExtractor:

    def __init__(self, hg38_fasta: str):
        log.info(f"Loading hg38 reference: {hg38_fasta}")
        self.fasta = Fasta(
            hg38_fasta,
            indexname="/kaggle/working/hg38.fa.fai",  # ← write index here
            rebuild=True
        )

    def get_sequence(self, chrom: str, start: int, end: int) -> str:
        chrom_key = chrom if chrom in self.fasta else f"chr{chrom}"
        try:
            return str(self.fasta[chrom_key][start:end]).upper()
        except Exception:
            return ""


def beta_to_token(beta: float, rng: np.random.Generator) -> str:
    if np.isnan(beta):
        return "<um>"
    if beta > BETA_HIGH:
        return "<m>"
    if beta < BETA_LOW:
        return "<um>"
    return "<m>" if rng.random() < beta else "<um>"


def build_methylated_string(
    sequence: str,
    cpg_positions: List[int],
    beta_values: Dict[int, float],
    rng: np.random.Generator,
) -> List[str]:
    tokens  = []
    cpg_set = set(cpg_positions)
    for i, nuc in enumerate(sequence):
        if i in cpg_set:
            tokens.append(beta_to_token(beta_values.get(i, np.nan), rng))
            tokens.append("G")
        elif nuc in "ATCG":
            tokens.append(nuc)
        else:
            tokens.append("N")
    return tokens


def find_cpg_positions(sequence: str) -> List[int]:
    seq, positions = sequence.upper(), []
    for i in range(len(seq) - 1):
        if seq[i] == "C" and seq[i + 1] == "G":
            positions.append(i)
    return positions


# ---------------------------------------------------------------------------
# Step 6 — Fragment Generation
# ---------------------------------------------------------------------------

class FragmentGenerator:

    def __init__(self, is_tumor: bool, rng: np.random.Generator):
        self.is_tumor   = is_tumor
        self.rng        = rng
        self._mono_mean = FRAG_TUMOR_MONO   if is_tumor else FRAG_HEALTHY_MONO
        self._mono_sd   = FRAG_SD_TUMOR     if is_tumor else FRAG_SD_HEALTHY

    def sample_fragment_length(self) -> int:
        if self.rng.random() < FRAG_MONO_RATIO:
            length = int(self.rng.normal(self._mono_mean, self._mono_sd))
            if self.is_tumor:
                length = int(np.clip(length, FRAG_TUMOR_MONO_MIN, FRAG_TUMOR_MONO_MAX))
            else:
                length = int(np.clip(length, FRAG_MIN, FRAG_MAX))
        else:
            length = int(np.clip(self.rng.normal(FRAG_DINU, 15), FRAG_MIN, FRAG_MAX))
        return length

    def fragment_token_stream(
        self,
        all_tokens: List[str],
        chrom: str,
        window_start: int,
    ) -> List[Fragment]:
        fragments, pos, total = [], 0, len(all_tokens)
        while pos < total:
            target_end  = pos + self.sample_fragment_length()
            actual_end  = min(self._find_gc_cut(all_tokens, target_end, window=10), total)
            frag_tokens = all_tokens[pos:actual_end]
            if len(frag_tokens) < FRAG_MIN:
                break
            nt_before     = sum(1 for t in all_tokens[:pos] if len(t) == 1)
            genomic_start = window_start + nt_before
            fragments.append(Fragment(
                tokens=["<cfdna>"] + frag_tokens + ["</cfdna>"],
                fragment_length=len(frag_tokens),
                cpg_count=frag_tokens.count("<m>"),
                strand="+" if self.rng.random() > 0.5 else "-",
                genomic_start=genomic_start,
                chromosome=chrom,
            ))
            pos = actual_end
        return fragments

    def _find_gc_cut(self, tokens: List[str], target: int, window: int = 10) -> int:
        lo, hi = max(0, target - window), min(len(tokens), target + window)
        if lo >= hi:
            return target
        gc_tokens  = {"G", "C", "<m>", "<um>"}
        best_pos, best_score = target, -1
        for i in range(lo, hi):
            score = sum(1 for t in tokens[max(0, i - 3): min(len(tokens), i + 3)]
                        if t in gc_tokens)
            if score > best_score:
                best_score, best_pos = score, i
        return best_pos

    @staticmethod
    def filter_by_cpg(fragments: List[Fragment]) -> List[Fragment]:
        return [f for f in fragments if MIN_CPG_PER_FRAG <= f.cpg_count <= MAX_CPG_PER_FRAG]


# ---------------------------------------------------------------------------
# Step 7 — ctDNA Signal Mixing
# ---------------------------------------------------------------------------

def mix_ctdna_fragments(
    tumor_fragments: List[Fragment],
    healthy_fragments: List[Fragment],
    rng: np.random.Generator,
    enriched: bool = True,
) -> Tuple[List[Fragment], float, float]:
    """
    FIX-2: returns (mixed, raw_ctdna_fraction, enriched_ctdna_fraction).
    Both values are stored in the JSON so metadata matches actual data.
    """
    if not tumor_fragments:
        return healthy_fragments, 0.0, 0.0

    raw_ctdna     = float(rng.uniform(CTDNA_MEDIAN, CTDNA_RANGE_MAX))
    enriched_frac = float(rng.uniform(ENRICHED_CTDNA_MIN, ENRICHED_CTDNA_MAX)) \
                    if enriched else raw_ctdna

    n_total   = len(tumor_fragments) + len(healthy_fragments)
    n_tumor   = max(1, int(n_total * enriched_frac))
    n_healthy = n_total - n_tumor

    sel_tumor   = rng.choice(len(tumor_fragments),   size=min(n_tumor,   len(tumor_fragments)),   replace=False)
    sel_healthy = rng.choice(len(healthy_fragments), size=min(n_healthy, len(healthy_fragments)), replace=True)

    mixed = [tumor_fragments[i] for i in sel_tumor] + \
            [healthy_fragments[i] for i in sel_healthy]
    rng.shuffle(mixed)
    return mixed, raw_ctdna, enriched_frac


# ---------------------------------------------------------------------------
# Step 8 — JSON Serialization
# ---------------------------------------------------------------------------

def serialize_patient(
    label: int,
    patient_id: str,
    fragments: List[Fragment],
    raw_ctdna_fraction: float,
    enriched_ctdna_fraction: float,
    n_fragments_target: int = 750,
    rng: Optional[np.random.Generator] = None,
) -> Dict:
    """
    FIX-2: JSON stores both raw_ctdna_fraction and enriched_ctdna_fraction.
    """
    if rng is None:
        rng = np.random.default_rng()

    if len(fragments) > n_fragments_target:
        idxs      = rng.choice(len(fragments), size=n_fragments_target, replace=False)
        fragments = [fragments[i] for i in idxs]

    regions: Dict[str, Dict] = {}
    region_counter = 0
    chrom_groups: Dict[str, List[Fragment]] = {}
    for frag in fragments:
        chrom_groups.setdefault(frag.chromosome, []).append(frag)

    for chrom, chrom_frags in sorted(chrom_groups.items()):
        chrom_frags  = sorted(chrom_frags, key=lambda f: f.genomic_start)
        window_start = chrom_frags[0].genomic_start if chrom_frags else 0
        window_frags: List[Fragment] = []

        for frag in chrom_frags:
            if frag.genomic_start - window_start > WINDOW_BP and window_frags:
                regions[f"region_{region_counter}"] = _make_region(chrom, window_start, window_frags)
                region_counter += 1
                window_start = frag.genomic_start
                window_frags = []
            window_frags.append(frag)

        if window_frags:
            regions[f"region_{region_counter}"] = _make_region(chrom, window_start, window_frags)
            region_counter += 1

    return {
        "label":                   label,
        "subtype":                 LABEL_MAP[label],
        "patient_id":              patient_id,
        "raw_ctdna_fraction":      round(raw_ctdna_fraction, 9),
        "enriched_ctdna_fraction": round(enriched_ctdna_fraction, 4),
        "n_fragments":             len(fragments),
        "regions":                 regions,
    }


def _make_region(chrom: str, window_start: int, frags: List[Fragment]) -> Dict:
    return {
        "chromosome": int(chrom.lstrip("chr")) if chrom.lstrip("chr").isdigit() else chrom,
        "genomic_start": window_start,
        "fragments": [_frag_to_dict(f) for f in frags],
    }


def _frag_to_dict(f: Fragment) -> Dict:
    return {
        "tokens":          f.tokens,
        "fragment_length": f.fragment_length,
        "cpg_count":       f.cpg_count,
        "strand":          f.strand,
    }


# ---------------------------------------------------------------------------
# Manifest Loading Helper
# ---------------------------------------------------------------------------

def load_illumina_manifest(manifest_path: str) -> pd.DataFrame:
    """
    Load probe coordinates for the Illumina HumanMethylation450 array.

    Tries sources in order:
      1. Local CSV file at manifest_path (if it exists)
      2. GPL13534 platform table fetched directly from NCBI GEO (no download needed)

    Returns:
        DataFrame indexed by cg_id with columns ['chromosome', 'start', 'end'].
    """
    import urllib.request, gzip, io as _io

    # Option 1: local CSV
    if os.path.exists(manifest_path):
        log.info(f"Loading 450K manifest from local file: {manifest_path}")
        df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
        df = df[["IlmnID", "CHR", "MAPINFO"]].dropna()
        df.columns = ["cg_id", "chromosome", "start"]
        df["start"] = df["start"].astype(int)
        df["end"]   = df["start"] + 1
        return df.set_index("cg_id")

    # Option 2: fetch GPL13534 from NCBI GEO
    log.info(
        "Manifest CSV not found — fetching GPL13534 probe coordinates "
        "from NCBI GEO (one-time ~10 MB download) ..."
    )
    url = "https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL13nnn/GPL13534/soft/GPL13534_family.soft.gz"
    try:
        with urllib.request.urlopen(url, timeout=120) as resp:
            raw = resp.read()
    except Exception as e:
        raise RuntimeError(
            f"Could not fetch GPL13534 from NCBI GEO: {e}\n"
            "Please download HumanMethylation450_15017482_v1-2.csv from Illumina "
            "and add it as a Kaggle dataset at /kaggle/input/illumina-manifest/."
        )

    log.info("Parsing GPL13534 platform table ...")
    records  = []
    in_table = False
    idx_id = idx_chr = idx_pos = None

    with gzip.open(_io.BytesIO(raw), "rt", encoding="latin-1") as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith("!platform_table_begin"):
                in_table   = True
                header     = next(fh).rstrip("\n").split("\t")
                cols_lower = [c.lower() for c in header]
                idx_id  = cols_lower.index("id")
                idx_chr = next(i for i, c in enumerate(cols_lower) if "chr" in c)
                idx_pos = next(i for i, c in enumerate(cols_lower) if "mapinfo" in c)
                continue
            if line.startswith("!platform_table_end"):
                break
            if not in_table:
                continue
            parts = line.split("\t")
            if len(parts) <= max(idx_id, idx_chr, idx_pos):
                continue
            cg_id = parts[idx_id].strip()
            chrom = parts[idx_chr].strip()
            pos   = parts[idx_pos].strip()
            if not cg_id.startswith("cg") or not chrom or not pos:
                continue
            try:
                records.append((cg_id, chrom, int(float(pos))))
            except ValueError:
                continue

    if not records:
        raise RuntimeError(
            "GPL13534 parsed 0 records — platform table format may have changed. "
            "Please supply the Illumina manifest CSV manually."
        )

    df = pd.DataFrame(records, columns=["cg_id", "chromosome", "start"])
    df["end"] = df["start"] + 1
    df = df.set_index("cg_id")
    log.info(f"GPL13534 manifest loaded: {len(df):,} probes")
    return df


# ---------------------------------------------------------------------------
# Top-Level Orchestrator
# ---------------------------------------------------------------------------

class SyntheticCfDNAPipeline:

    def __init__(self, hg38_fasta: str, manifest_path: str, output_dir: str, seed: int = 42):
        self.seq_extractor = SequenceExtractor(hg38_fasta)
        self.manifest      = load_illumina_manifest(manifest_path)
        self.output_dir    = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.rng = np.random.default_rng(seed)
        log.info(f"Output directory: {self.output_dir}")

    def run(
        self,
        matrices: Dict[int, pd.DataFrame],
        dmrs_by_label: Dict[int, List[DMR]],
        healthy_matrix: pd.DataFrame,
        n_frags_target: int = 750,
    ) -> None:
        log.info("Pre-generating healthy fragment pool ...")
        healthy_frag_pool = self._generate_healthy_pool(healthy_matrix, dmrs_by_label)

        self._process_label_0(healthy_matrix, healthy_frag_pool, n_frags_target)

        for label, matrix in matrices.items():
            if label == 0:
                continue
            dmrs = dmrs_by_label.get(label, [])
            if not dmrs:
                log.warning(f"No DMRs for label {label}, skipping.")
                continue
            log.info(f"\n{'='*60}\nLabel={label} ({LABEL_MAP[label]}) — "
                     f"{matrix.shape[1]} patients, {len(dmrs)} DMRs\n{'='*60}")
            self._process_label(label, matrix, dmrs, healthy_frag_pool, n_frags_target)

        log.info("Pipeline complete.")

    def _generate_healthy_pool(
        self,
        healthy_matrix: pd.DataFrame,
        dmrs_by_label: Dict[int, List[DMR]],
    ) -> List[Fragment]:
        """
        FIX-3: Per-probe beta values looked up by cg_id (not column-wide mean).
        """
        all_dmrs = [d for dl in dmrs_by_label.values() for d in dl]
        unique_dmrs: Dict[Tuple, DMR] = {}
        for d in all_dmrs:
            key = (d.chromosome, d.start, d.end)
            if key not in unique_dmrs:
                unique_dmrs[key] = d

        n_sample    = min(50, healthy_matrix.shape[1])
        sample_cols = self.rng.choice(healthy_matrix.columns, size=n_sample, replace=False)

        pool: List[Fragment] = []
        gen = FragmentGenerator(is_tumor=False, rng=self.rng)

        for (chrom, start, end), dmr in tqdm(unique_dmrs.items(), desc="Healthy pool"):
            seq = self.seq_extractor.get_sequence(chrom, start, end)
            if not seq:
                continue
            cpg_pos = find_cpg_positions(seq)
            if len(cpg_pos) < MIN_CPGS:
                continue

            for col in sample_cols:
                patient_betas = healthy_matrix[col]
                # FIX-3: look up actual per-probe beta by cg_id
                betas: Dict[int, float] = {}
                for k, cg_id in enumerate(dmr.cpg_ids):
                    if cg_id in patient_betas.index and k < len(cpg_pos):
                        val = patient_betas[cg_id]
                        if not np.isnan(val):
                            betas[cpg_pos[k]] = float(val)

                tokens = build_methylated_string(seq, cpg_pos, betas, self.rng)
                frags  = gen.fragment_token_stream(tokens, chrom, start)
                pool.extend(FragmentGenerator.filter_by_cpg(frags))

        log.info(f"  Healthy pool: {len(pool):,} fragments")
        return pool

    def _process_label_0(
        self,
        healthy_matrix: pd.DataFrame,
        healthy_pool: List[Fragment],
        n_frags_target: int,
    ) -> None:
        log.info(f"Generating label=0 (Healthy) — {healthy_matrix.shape[1]} patients")
        label_dir = self.output_dir / "label_0_Healthy"
        label_dir.mkdir(exist_ok=True)

        for patient_id in tqdm(healthy_matrix.columns, desc="Healthy patients"):
            n_frags = int(self.rng.integers(FRAGS_PER_PATIENT_MIN, FRAGS_PER_PATIENT_MAX))
            idxs    = self.rng.choice(len(healthy_pool), size=min(n_frags, len(healthy_pool)), replace=True)
            frags   = [healthy_pool[i] for i in idxs]
            patient_json = serialize_patient(
                label=0, patient_id=str(patient_id), fragments=frags,
                raw_ctdna_fraction=0.0, enriched_ctdna_fraction=0.0,
                n_fragments_target=n_frags_target, rng=self.rng,
            )
            with open(label_dir / f"{patient_id}.json", "w") as f:
                json.dump(patient_json, f, separators=(",", ":"))

    def _process_label(
        self,
        label: int,
        matrix: pd.DataFrame,
        dmrs: List[DMR],
        healthy_pool: List[Fragment],
        n_frags_target: int,
    ) -> None:
        label_dir = self.output_dir / f"label_{label}_{LABEL_MAP[label]}"
        label_dir.mkdir(exist_ok=True)
        gen = FragmentGenerator(is_tumor=True, rng=self.rng)

        for patient_id in tqdm(matrix.columns, desc=f"Label {label}"):
            patient_betas = matrix[patient_id]
            tumor_frags: List[Fragment] = []

            for dmr in dmrs:
                seq = self.seq_extractor.get_sequence(dmr.chromosome, dmr.start, dmr.end)
                if not seq:
                    continue
                cpg_pos = find_cpg_positions(seq)
                if len(cpg_pos) < MIN_CPGS:
                    continue

                betas: Dict[int, float] = {}
                for k, cg_id in enumerate(dmr.cpg_ids):
                    if cg_id in patient_betas.index and k < len(cpg_pos):
                        val = patient_betas[cg_id]
                        if not np.isnan(val):
                            betas[cpg_pos[k]] = float(val)

                tokens = build_methylated_string(seq, cpg_pos, betas, self.rng)
                frags  = gen.fragment_token_stream(tokens, dmr.chromosome, dmr.start)
                tumor_frags.extend(FragmentGenerator.filter_by_cpg(frags))

            # FIX-2: unpack all three return values
            mixed, raw_ctdna, enriched_ctdna = mix_ctdna_fragments(
                tumor_frags, healthy_pool, self.rng, enriched=True
            )
            patient_json = serialize_patient(
                label=label, patient_id=str(patient_id), fragments=mixed,
                raw_ctdna_fraction=raw_ctdna, enriched_ctdna_fraction=enriched_ctdna,
                n_fragments_target=n_frags_target, rng=self.rng,
            )
            with open(label_dir / f"{patient_id}.json", "w") as f:
                json.dump(patient_json, f, separators=(",", ":"))


# ---------------------------------------------------------------------------
# Entry Point
# ---------------------------------------------------------------------------
# argparse intentionally not used — Kaggle kernels inject their own sys.argv.
# ---------------------------------------------------------------------------

BASE = "/kaggle/input/datasets/kwarkp/model-5-dataset-prepration/Model 5 (dataset)"

CONFIG = {
    "tcga_gbm_dir":         f"{BASE}/TCGA-GBM ( Methylation for CfDNA) Model 5",
    "tcga_lgg_dir":         f"{BASE}/TCGA-LGG (Methylation for CfDNA) Model 5",

    # DMG: GSE161944 only — no GSE50022, no DIPG
    "dmg_matrix_gse161944": f"{BASE}/GSE161944 (DMG H3K27M)/GSE161944_Matrix_raw.txt",

    # Healthy baseline: GSE40279 (whole-blood methylation)
    "healthy_matrix":       f"{BASE}/GSE40279 (Healthy Baseline)/GSE40279_beta.txt",

    "hg38_fasta":           "/kaggle/input/datasets/kwarkp/hg38-dataset/hg38.fa",
    "manifest":             "/kaggle/input/datasets/kwarkp/illumina-manifest-dataset/GPL13534_HumanMethylation450_15017482_v.1.1.csv",
    "output_dir":           "/kaggle/working/synthetic_cfdna_output",
    "seed":                 42,
    "na_threshold":         0.70,
    "frags_per_patient":    750,
}


def main(cfg: dict = CONFIG) -> None:
    """
    Run the full 4-class synthetic cfDNA pipeline.

        main()                        # CONFIG defaults
        main({**CONFIG, "seed": 99})  # override keys
    """
    # ── Step 1: Load raw data ──────────────────────────────────────────────
    loader     = DataLoader()
    gbm_matrix = loader.load_tcga_per_patient(cfg["tcga_gbm_dir"], label=1, na_threshold=cfg["na_threshold"])
    lgg_matrix = loader.load_tcga_per_patient(cfg["tcga_lgg_dir"], label=2, na_threshold=cfg["na_threshold"])
    log.info(f"LGG loaded as single class: {lgg_matrix.shape[1]} patients")

    dmg_matrix_real = loader.load_geo_matrix(cfg["dmg_matrix_gse161944"])
    log.info(f"DMG matrix (GSE161944 only): {dmg_matrix_real.shape[1]} real samples")

    healthy_matrix = loader.load_geo_matrix(cfg["healthy_matrix"])
    log.info(f"Healthy matrix (GSE40279): {healthy_matrix.shape[1]} samples")

    # ── Step 2: Common CpG intersection ───────────────────────────────────
    common_cpgs     = compute_common_cpg_space(gbm_matrix, lgg_matrix, dmg_matrix_real, healthy_matrix)
    gbm_matrix      = gbm_matrix.loc[common_cpgs]
    lgg_matrix      = lgg_matrix.loc[common_cpgs]
    dmg_matrix_real = dmg_matrix_real.reindex(common_cpgs)
    healthy_matrix  = healthy_matrix.reindex(common_cpgs)

    # ── Step 3: DMG augmentation ───────────────────────────────────────────
    dmg_aug = DMGAugmentor()
    dmg_aug.fit(dmg_matrix_real)
    dmg_matrix_augmented = dmg_aug.sample(n=DMG_AUGMENT_TARGET).reindex(common_cpgs)

    # ── Step 4: DMR identification ─────────────────────────────────────────
    manifest = load_illumina_manifest(cfg["manifest"])
    manifest = manifest.loc[manifest.index.isin(common_cpgs)]

    all_matrices = {
        0: healthy_matrix,
        1: gbm_matrix,
        2: lgg_matrix,
        3: dmg_matrix_augmented,
    }
    dmrs_by_label = DMRFinder(cpg_manifest=manifest).find_dmrs(all_matrices)

    # ── Steps 5-8: Fragment generation + serialization ─────────────────────
    pipeline = SyntheticCfDNAPipeline(
        hg38_fasta=cfg["hg38_fasta"],
        manifest_path=cfg["manifest"],
        output_dir=cfg["output_dir"],
        seed=cfg["seed"],
    )
    pipeline.run(
        matrices=all_matrices,
        dmrs_by_label=dmrs_by_label,
        healthy_matrix=healthy_matrix,
        n_frags_target=cfg["frags_per_patient"],
    )


if __name__ == "__main__":
    main()

08:58:29  INFO  Loading TCGA data from /kaggle/input/datasets/kwarkp/model-5-dataset-prepration/Model 5 (dataset)/TCGA-GBM ( Methylation for CfDNA) Model 5 (label=1)
TCGA label=1: 100%|██████████| 450/450 [01:49<00:00,  4.10it/s]
09:01:49  INFO    Raw shape: (488027, 450)
09:01:51  INFO    After NA filter (70%): (401421, 450) (86606 probes dropped)
09:02:47  INFO    Beta mean=0.484  std=0.379
09:02:49  INFO  Loading TCGA data from /kaggle/input/datasets/kwarkp/model-5-dataset-prepration/Model 5 (dataset)/TCGA-LGG (Methylation for CfDNA) Model 5 (label=2)
TCGA label=2: 100%|██████████| 534/534 [05:35<00:00,  1.59it/s]
09:08:58  INFO    Raw shape: (486427, 534)
09:09:01  INFO    After NA filter (70%): (420162, 534) (66265 probes dropped)
09:09:55  INFO    Beta mean=0.551  std=0.387
09:10:00  INFO  LGG loaded as single class: 534 patients
09:10:00  INFO  Loading GEO matrix: /kaggle/input/datasets/kwarkp/model-5-dataset-prepration/Model 5 (dataset)/GSE161944 (DMG H3K27M)/GSE161944_Matrix_r

In [ ]:
"""
FIX CELL — corrects three rulebook violations in pipeline.py
Run this cell AFTER the main pipeline cell (it monkey-patches the same
names so all downstream calls automatically use the corrected versions).

1. mix_ctdna_fragments  — tumor fraction was diluted by global pool sizing.
                           Now scales healthy count off the tumor count,
                           so enriched_frac is actually honored.
2. filter_by_cpg         — was counting only <m> tokens. Now counts total
                           CpG loci (<m> + <um>), matching Rule 8.
3. fragment_token_stream — last fragment per window was truncated to
                           "whatever's left" instead of being a real draw
                           from the bimodal length model. Now the tail
                           fragment is merged into its neighbor instead of
                           being kept as a malformed remainder, and tiny
                           leftover scraps are dropped rather than emitted.
"""

from typing import List, Tuple


# ---------------------------------------------------------------------------
# FIX 1 — Global Pool Dilution in mix_ctdna_fragments
# ---------------------------------------------------------------------------
# OLD: n_total = len(tumor) + len(healthy); n_tumor = n_total * enriched_frac
#      -> tumor count is capped by availability (no replacement) while
#         healthy backfills with replacement, so realized tumor share
#         silently collapses below enriched_frac whenever the healthy
#         pool is large.
# NEW: anchor on the tumor fragments actually available, derive the
#      healthy count FROM that, so the ratio in the output matches
#      enriched_frac regardless of pool sizes.

def mix_ctdna_fragments(
    tumor_fragments: List["Fragment"],
    healthy_fragments: List["Fragment"],
    rng,
    enriched: bool = True,
) -> Tuple[List["Fragment"], float, float]:
    """
    FIX-2 (original) + FIX-3 (this patch): returns
    (mixed, raw_ctdna_fraction, enriched_ctdna_fraction).

    Healthy/tumor counts are now derived FROM the tumor pool, not from
    the combined pool size, so enriched_frac is the actual realized
    ratio rather than an upper bound that healthy fragments can drown out.
    """
    if not tumor_fragments:
        return healthy_fragments, 0.0, 0.0

    raw_ctdna     = float(rng.uniform(CTDNA_MEDIAN, CTDNA_RANGE_MAX))
    enriched_frac = float(rng.uniform(ENRICHED_CTDNA_MIN, ENRICHED_CTDNA_MAX)) \
                    if enriched else raw_ctdna

    n_tumor   = len(tumor_fragments)
    n_healthy = max(1, int(round(n_tumor * (1.0 - enriched_frac) / enriched_frac)))
    n_healthy = min(n_healthy, max(len(healthy_fragments), 1))

    sel_tumor   = rng.choice(n_tumor, size=n_tumor, replace=False)
    sel_healthy = rng.choice(
        len(healthy_fragments),
        size=min(n_healthy, len(healthy_fragments)),
        replace=len(healthy_fragments) < n_healthy,
    )

    mixed = [tumor_fragments[i] for i in sel_tumor] + \
            [healthy_fragments[i] for i in sel_healthy]
    rng.shuffle(mixed)

    realized_frac = n_tumor / len(mixed) if mixed else 0.0
    return mixed, raw_ctdna, realized_frac


# ---------------------------------------------------------------------------
# FIX 2 — CpG Filtering Selection Bias in filter_by_cpg
# ---------------------------------------------------------------------------
# OLD: cpg_count = frag_tokens.count("<m>")  -> only methylated CpGs counted,
#      so structurally valid but hypomethylated fragments get purged.
# NEW: count total CpG loci = <m> + <um>, matching Rule 8's definition of
#      "CpG site," not "methylated CpG site."

def filter_by_cpg(fragments: List["Fragment"]) -> List["Fragment"]:
    def total_cpg_loci(frag) -> int:
        return frag.tokens.count("<m>") + frag.tokens.count("<um>")
    return [f for f in fragments
            if MIN_CPG_PER_FRAG <= total_cpg_loci(f) <= MAX_CPG_PER_FRAG]


# Patch onto FragmentGenerator so gen.filter_by_cpg(...) keeps working
# wherever it's called as a static/bound method.
FragmentGenerator.filter_by_cpg = staticmethod(filter_by_cpg)


# ---------------------------------------------------------------------------
# FIX 3 — Window Boundary Truncation in fragment_token_stream
# ---------------------------------------------------------------------------
# OLD: while pos < total: cut at sampled length: the LAST fragment per
#      window is just "whatever tokens remain," not a real draw from the
#      bimodal length model, silently violating Rule 5 for one fragment
#      per window.
# NEW: same sampling/cut logic, but after the loop, if the final fragment
#      is short relative to a real draw (i.e. it was truncated by running
#      out of window rather than by an actual sampled+GC-biased cut), it
#      gets merged into the previous fragment instead of being kept as a
#      separately-counted, distribution-violating fragment. If there's no
#      previous fragment to merge into (window itself was too short), it's
#      dropped rather than emitted as a malformed sample.

def fragment_token_stream(
    self,
    all_tokens: List[str],
    chrom: str,
    window_start: int,
) -> List["Fragment"]:
    fragments, pos, total = [], 0, len(all_tokens)
    while pos < total:
        sampled_len = self.sample_fragment_length()
        target_end  = pos + sampled_len
        actual_end  = min(self._find_gc_cut(all_tokens, target_end, window=10), total)
        frag_tokens = all_tokens[pos:actual_end]
        if len(frag_tokens) < FRAG_MIN:
            break
        nt_before     = sum(1 for t in all_tokens[:pos] if len(t) == 1)
        genomic_start = window_start + nt_before

        is_window_truncated = (actual_end == total) and (len(frag_tokens) < sampled_len * 0.8)

        if is_window_truncated and fragments:
            # Merge the truncated remainder into the previous fragment
            # instead of emitting it as a separate, distribution-violating
            # fragment. Strip the previous </cfdna> close tag before
            # appending the remainder + a fresh close tag.
            prev = fragments[-1]
            merged_tokens = prev.tokens[:-1] + frag_tokens + ["</cfdna>"]
            fragments[-1] = Fragment(
                tokens=merged_tokens,
                fragment_length=prev.fragment_length + len(frag_tokens),
                cpg_count=prev.cpg_count + frag_tokens.count("<m>"),
                strand=prev.strand,
                genomic_start=prev.genomic_start,
                chromosome=prev.chromosome,
            )
            pos = actual_end
            continue
        elif is_window_truncated and not fragments:
            # No prior fragment to merge into; drop the malformed remainder
            # rather than emit a fragment that violates the length model.
            break

        fragments.append(Fragment(
            tokens=["<cfdna>"] + frag_tokens + ["</cfdna>"],
            fragment_length=len(frag_tokens),
            cpg_count=frag_tokens.count("<m>"),
            strand="+" if self.rng.random() > 0.5 else "-",
            genomic_start=genomic_start,
            chromosome=chrom,
        ))
        pos = actual_end
    return fragments


# Patch onto FragmentGenerator so gen.fragment_token_stream(...) calls
# automatically use the corrected version.
FragmentGenerator.fragment_token_stream = fragment_token_stream

print("Patched: mix_ctdna_fragments, filter_by_cpg, FragmentGenerator.fragment_token_stream")

In [4]:
"""
validate_dataset.py — Post-generation QC for the synthetic cfDNA dataset.
Checks all biological constraints from the rulebook before training M5.

Run after pipeline.py completes:
    python validate_dataset.py --output_dir ./synthetic_cfdna_output
"""

import os
import json
import glob
import logging
from pathlib import Path
from collections import defaultdict
from typing import Dict, List

import numpy as np
import pandas as pd
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)s  %(message)s")
log = logging.getLogger(__name__)

# Tolerances (mirroring pipeline constants)
FRAG_MIN = 100
FRAG_MAX = 400
MIN_CPG_PER_FRAG = 2
MAX_CPG_PER_FRAG = 8
FRAGS_PER_PATIENT_MIN = 500
FRAGS_PER_PATIENT_MAX = 1000


def load_all_patients(output_dir: str) -> List[Dict]:
    files = glob.glob(os.path.join(output_dir, "**", "*.json"), recursive=True)
    patients = []
    for fpath in tqdm(files, desc="Loading JSONs"):
        with open(fpath) as f:
            patients.append(json.load(f))
    return patients


def validate(output_dir: str) -> None:
    patients = load_all_patients(output_dir)
    log.info(f"Total patients: {len(patients)}")

    by_label: Dict[int, List[Dict]] = defaultdict(list)
    for p in patients:
        by_label[p["label"]].append(p)

    # --- Class balance report ---
    log.info("\n=== Class Distribution ===")
    for lbl in sorted(by_label):
        log.info(f"  Label {lbl} ({by_label[lbl][0]['subtype']}): {len(by_label[lbl])} patients")

    # --- Per-patient fragment checks ---
    log.info("\n=== Fragment-level Checks ===")
    errors = []

    for p in tqdm(patients, desc="Validating patients"):
        pid = p.get("patient_id", "unknown")
        n_frags = p.get("n_fragments", 0)

        all_frags = []
        for region in p["regions"].values():
            all_frags.extend(region["fragments"])

        if not all_frags:
            errors.append(f"{pid}: 0 fragments")
            continue

        for frag in all_frags:
            fl = frag["fragment_length"]
            cg = frag["cpg_count"]

            if not (FRAG_MIN <= fl <= FRAG_MAX):
                errors.append(f"{pid}: fragment_length={fl} out of [{FRAG_MIN},{FRAG_MAX}]")
            if not (MIN_CPG_PER_FRAG <= cg <= MAX_CPG_PER_FRAG):
                errors.append(f"{pid}: cpg_count={cg} out of [{MIN_CPG_PER_FRAG},{MAX_CPG_PER_FRAG}]")

            tokens = frag["tokens"]
            if tokens[0] != "<cfdna>" or tokens[-1] != "</cfdna>":
                errors.append(f"{pid}: missing <cfdna> wrapper")
                break
            if frag["strand"] not in ("+", "-"):
                errors.append(f"{pid}: invalid strand '{frag['strand']}'")

    # --- Fragment length distribution per label ---
    log.info("\n=== Fragment Length Distributions ===")
    for lbl in sorted(by_label):
        lengths = []
        for p in by_label[lbl]:
            for region in p["regions"].values():
                for frag in region["fragments"]:
                    lengths.append(frag["fragment_length"])
        if lengths:
            arr = np.array(lengths)
            log.info(
                f"  Label {lbl}: mean={arr.mean():.1f}  "
                f"median={np.median(arr):.1f}  "
                f"std={arr.std():.1f}  "
                f"[{arr.min()}, {arr.max()}]"
            )

    # --- ctDNA fraction range ---
    log.info("\n=== ctDNA Fraction Check ===")
    for lbl in sorted(by_label):
        raw_fracs      = [p["raw_ctdna_fraction"]      for p in by_label[lbl]]
        enriched_fracs = [p["enriched_ctdna_fraction"] for p in by_label[lbl]]
        log.info(
            f"  Label {lbl} raw:      "
            f"mean={np.mean(raw_fracs):.2e}  "
            f"min={np.min(raw_fracs):.2e}  "
            f"max={np.max(raw_fracs):.2e}"
        )
        log.info(
            f"  Label {lbl} enriched: "
            f"mean={np.mean(enriched_fracs):.4f}  "
            f"min={np.min(enriched_fracs):.4f}  "
            f"max={np.max(enriched_fracs):.4f}"
        )

    # --- Token vocabulary check ---
    log.info("\n=== Token Vocabulary Check ===")
    valid_tokens = {"<cfdna>", "</cfdna>", "<m>", "<um>", "A", "T", "G", "C", "N"}
    vocab_errors = 0
    for p in patients[:100]:  # sample check
        for region in p["regions"].values():
            for frag in region["fragments"]:
                for tok in frag["tokens"]:
                    if tok not in valid_tokens:
                        vocab_errors += 1
    log.info(f"  Invalid tokens in first 100 patients: {vocab_errors}")

    # --- Summary ---
    log.info(f"\n=== Validation Summary ===")
    log.info(f"  Total errors: {len(errors)}")
    if errors:
        for e in errors[:20]:
            log.warning(f"  {e}")
        if len(errors) > 20:
            log.warning(f"  ... and {len(errors) - 20} more")
    else:
        log.info("  All checks passed ✓")

    # --- Export summary CSV ---
    summary_rows = []
    for lbl in sorted(by_label):
        all_lengths = []
        for p in by_label[lbl]:
            for region in p["regions"].values():
                for frag in region["fragments"]:
                    all_lengths.append(frag["fragment_length"])
        summary_rows.append({
            "label":               lbl,
            "subtype":             by_label[lbl][0]["subtype"],
            "n_patients":          len(by_label[lbl]),
            "total_fragments":     len(all_lengths),
            "mean_frag_length":    round(np.mean(all_lengths), 1) if all_lengths else 0,
            "median_frag_length":  round(np.median(all_lengths), 1) if all_lengths else 0,
        })
    pd.DataFrame(summary_rows).to_csv(
        Path(output_dir) / "dataset_summary.csv", index=False
    )
    log.info(f"  Summary written to {Path(output_dir) / 'dataset_summary.csv'}")


# ── Run directly in Kaggle notebook ──
OUTPUT_DIR = "/kaggle/working/synthetic_cfdna_output"
validate(OUTPUT_DIR)

Loading JSONs: 100%|██████████| 1840/1840 [00:29<00:00, 62.46it/s]
09:24:00  INFO  Total patients: 1840
09:24:00  INFO  
=== Class Distribution ===
09:24:00  INFO    Label 0 (Healthy): 656 patients
09:24:00  INFO    Label 1 (GBM): 450 patients
09:24:00  INFO    Label 2 (LGG): 534 patients
09:24:00  INFO    Label 3 (DMG_H3K27M): 200 patients
09:24:00  INFO  
=== Fragment-level Checks ===
Validating patients: 100%|██████████| 1840/1840 [00:00<00:00, 2295.40it/s]
09:24:01  INFO  
=== Fragment Length Distributions ===
09:24:01  INFO    Label 0: mean=200.3  median=169.0  std=65.8  [100, 330]
09:24:01  INFO    Label 1: mean=200.3  median=169.0  std=65.9  [100, 340]
09:24:01  INFO    Label 2: mean=200.1  median=169.0  std=65.9  [100, 335]
09:24:01  INFO    Label 3: mean=200.3  median=169.0  std=66.0  [100, 330]
09:24:01  INFO  
=== ctDNA Fraction Check ===
09:24:01  INFO    Label 0 raw:      mean=0.00e+00  min=0.00e+00  max=0.00e+00
09:24:01  INFO    Label 0 enriched: mean=0.0000  min=0.0000 